# 03 Training Science — Learning Rate Scheduling

**Status:** Complete

**Learning outcome:** Understand and visualise step decay, cosine annealing, warmup, and OneCycleLR schedules — and when each is appropriate for training neural networks.

## Why This Matters

A fixed learning rate is rarely optimal: too high and training diverges, too low and it's painfully slow. LR schedules start high (for fast exploration) and decay to a small value (for fine-tuning near the optimum). Warmup prevents early instability when the model hasn't learned meaningful features yet. These schedules are a key hyperparameter in modern training recipes.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, OneCycleLR, LambdaLR

torch.manual_seed(42)
print("Libraries loaded.")

Libraries loaded.


## Schedule Visualisation

We simulate 100 epochs and plot the learning rate at each step for four common schedules.

In [2]:
def get_lr_schedule(scheduler_fn, epochs=100, steps_per_epoch=10):
    """Simulate a scheduler and record LR at each step."""
    model = nn.Linear(1, 1)
    opt = optim.SGD(model.parameters(), lr=0.1)
    sched = scheduler_fn(opt)
    lrs = []
    for epoch in range(epochs):
        for step in range(steps_per_epoch):
            lrs.append(opt.param_groups[0]['lr'])
            opt.step()
            if isinstance(sched, OneCycleLR):
                sched.step()
        if not isinstance(sched, OneCycleLR):
            sched.step()
    return lrs

total_steps = 100 * 10  # epochs * steps_per_epoch

schedules = {
    'StepLR (γ=0.5, every 30)': lambda opt: StepLR(opt, step_size=30, gamma=0.5),
    'CosineAnnealing': lambda opt: CosineAnnealingLR(opt, T_max=100),
    'Warmup + Cosine': lambda opt: LambdaLR(opt, lr_lambda=lambda epoch:
        min(1.0, (epoch + 1) / 10) * (0.5 * (1 + np.cos(np.pi * max(0, epoch - 10) / 90)))),
    'OneCycleLR': lambda opt: OneCycleLR(opt, max_lr=0.1, total_steps=total_steps),
}

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['red', 'blue', 'green', 'purple']

for (name, sched_fn), color in zip(schedules.items(), colors):
    lrs = get_lr_schedule(sched_fn)
    ax.plot(lrs, label=name, color=color, alpha=0.8)

ax.set_xlabel('Step (epoch × 10)')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules Comparison')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../assets/lr_schedules.png', dpi=100)
plt.show()
print("Schedule comparison saved.")

Schedule comparison saved.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_71711/2755865126.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding The Exploration-Exploitation Tradeoff**

**Plain language:** When you move to a new city, you first explore widely — trying many restaurants to find the good neighbourhoods. Once you've mapped the terrain, you exploit what you've learned by returning to your favourites and refining your choices. Learning rate scheduling does the same thing: start broad, then narrow down.

**Building intuition:** A high learning rate takes large steps through the loss landscape, which lets the optimiser jump over small local minima and explore different regions — this is *exploration*. But large steps also mean the optimiser overshoots good solutions and bounces around. A low learning rate takes small, precise steps that converge to the nearest minimum — this is *exploitation*. A schedule transitions from exploration (high LR) to exploitation (low LR) over the course of training, getting the benefits of both.

**Formal statement:** The update rule $\theta_{t+1} = \theta_t - \eta_t \nabla \mathcal{L}(\theta_t)$ shows that the step size is proportional to $\eta_t$. A schedule $\eta_t = f(t)$ is a monotonically decreasing function (in most cases) that balances the bias-variance tradeoff of the gradient estimate: early large $\eta_t$ escapes saddle points and sharp minima; late small $\eta_t$ converges to flat minima that generalise better (Keskar et al., 2017).

---

---
**Understanding Warmup: Preventing Early Instability**

**Plain language:** Imagine starting a car in freezing weather — you idle the engine for a minute to warm up before flooring the accelerator. If you gun it immediately, the cold engine stalls or damages itself. Neural network warmup does the same: it starts with a tiny learning rate and gradually increases to full speed as the model's internal statistics stabilise.

**Building intuition:** At the very start of training, all weights are random and gradients are noisy and unreliable — they point in roughly correct directions but with high variance. A large learning rate amplifies this noise, causing destructive updates that can send the model into bad regions of the loss landscape (or even cause NaN values). *Warmup* starts with $\eta \approx 0$ and linearly ramps to the target LR over the first few epochs. By the time the full LR kicks in, the model has learned enough structure that gradients are meaningful and stable.

**Formal statement:** Linear warmup over $T_w$ steps: $\eta_t = \eta_{\max} \cdot \frac{t}{T_w}$ for $t \leq T_w$. After warmup, the schedule transitions to the main decay (e.g., cosine): $\eta_t = \frac{\eta_{\max}}{2}\left(1 + \cos\left(\pi \frac{t - T_w}{T - T_w}\right)\right)$ for $t > T_w$. Warmup is especially critical for Transformers and large batch training, where initial gradient magnitudes can be orders of magnitude larger than steady-state values (Goyal et al., 2017).

---

## Training Comparison: Constant vs Cosine vs OneCycle

In [ ]:
# LR vs Loss Landscape — Conceptual Illustration
lr_fig, lr_axes = plt.subplots(1, 3, figsize=(15, 4))

# Create a loss landscape (combination of sine curves for hills + a global minimum)
x_landscape = np.linspace(0, 10, 500)
loss_landscape = 2.0 + np.sin(1.5 * x_landscape) + 0.5 * np.sin(3.5 * x_landscape) + \
                 0.3 * np.cos(5 * x_landscape) - 0.8 * np.exp(-0.5 * (x_landscape - 7.0)**2)

# Panel 1: Too High — large steps overshooting
ax = lr_axes[0]
ax.plot(x_landscape, loss_landscape, 'k-', linewidth=1.5)
ax.fill_between(x_landscape, loss_landscape, alpha=0.1, color='gray')
# Simulate large jumps that overshoot
positions = [1.0, 4.5, 1.8, 5.2, 2.5, 6.0]
for i in range(len(positions) - 1):
    x1, x2 = positions[i], positions[i + 1]
    y1 = np.interp(x1, x_landscape, loss_landscape)
    y2 = np.interp(x2, x_landscape, loss_landscape)
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.plot(x1, y1, 'ro', markersize=8)
ax.plot(positions[-1], np.interp(positions[-1], x_landscape, loss_landscape), 'ro', markersize=8)
ax.set_title('Too High LR', fontweight='bold', color='red')
ax.set_xlabel('Parameter space')
ax.set_ylabel('Loss')
ax.text(0.5, 0.02, 'Overshoots minima,\nbounces wildly', transform=ax.transAxes,
        ha='center', fontsize=9, style='italic', color='red')

# Panel 2: Too Low — stuck in local minimum
ax = lr_axes[1]
ax.plot(x_landscape, loss_landscape, 'k-', linewidth=1.5)
ax.fill_between(x_landscape, loss_landscape, alpha=0.1, color='gray')
# Tiny steps in a local minimum near x=3.4
local_min_x = 3.4
positions_low = [local_min_x - 0.3, local_min_x - 0.15, local_min_x - 0.05,
                 local_min_x + 0.02, local_min_x + 0.05]
for i in range(len(positions_low) - 1):
    x1, x2 = positions_low[i], positions_low[i + 1]
    y1 = np.interp(x1, x_landscape, loss_landscape)
    y2 = np.interp(x2, x_landscape, loss_landscape)
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='orange', lw=2))
    ax.plot(x1, y1, 'o', color='orange', markersize=8)
ax.plot(positions_low[-1], np.interp(positions_low[-1], x_landscape, loss_landscape),
        'o', color='orange', markersize=8)
# Mark the global minimum they can't reach
global_min_x = 7.0
ax.annotate('Global min\n(unreached)', xy=(global_min_x, np.interp(global_min_x, x_landscape, loss_landscape)),
            fontsize=8, ha='center', va='bottom', color='green',
            arrowprops=dict(arrowstyle='->', color='green'))
ax.set_title('Too Low LR', fontweight='bold', color='orange')
ax.set_xlabel('Parameter space')
ax.text(0.5, 0.02, 'Stuck in local minimum,\ntoo slow to escape', transform=ax.transAxes,
        ha='center', fontsize=9, style='italic', color='orange')

# Panel 3: Scheduled — large steps early, small steps late
ax = lr_axes[2]
ax.plot(x_landscape, loss_landscape, 'k-', linewidth=1.5)
ax.fill_between(x_landscape, loss_landscape, alpha=0.1, color='gray')
# Large jumps first, then small steps near global minimum
positions_sched = [1.0, 4.0, 6.0, 6.7, 6.9, 7.0]
for i in range(len(positions_sched) - 1):
    x1, x2 = positions_sched[i], positions_sched[i + 1]
    y1 = np.interp(x1, x_landscape, loss_landscape)
    y2 = np.interp(x2, x_landscape, loss_landscape)
    # Color gradient from blue (early/large) to green (late/small)
    t = i / (len(positions_sched) - 2)
    color = (0, t, 1 - t)  # blue -> green
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.plot(x1, y1, 'o', color=color, markersize=8)
ax.plot(positions_sched[-1], np.interp(positions_sched[-1], x_landscape, loss_landscape),
        '*', color='green', markersize=15, zorder=5)
ax.set_title('Scheduled LR', fontweight='bold', color='green')
ax.set_xlabel('Parameter space')
ax.text(0.5, 0.02, 'Large steps explore,\nsmall steps converge', transform=ax.transAxes,
        ha='center', fontsize=9, style='italic', color='green')

lr_fig.suptitle('Why Learning Rate Scheduling Matters', fontweight='bold', fontsize=13)
lr_fig.tight_layout()
lr_fig.savefig('../assets/lr_concept.png', dpi=100)
plt.show()
print("LR concept visualization saved to ../assets/lr_concept.png")

In [3]:
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset, random_split

digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)
dataset = TensorDataset(X, y)
n_train = int(0.7 * len(dataset))
train_ds, val_ds = random_split(dataset, [n_train, len(dataset) - n_train],
                                 generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256)

def train_with_schedule(sched_name, epochs=80):
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
    opt = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

    if sched_name == 'Constant':
        sched = LambdaLR(opt, lr_lambda=lambda e: 1.0)
    elif sched_name == 'Cosine':
        sched = CosineAnnealingLR(opt, T_max=epochs)
    elif sched_name == 'OneCycle':
        sched = OneCycleLR(opt, max_lr=0.1, epochs=epochs, steps_per_epoch=len(train_dl))

    criterion = nn.CrossEntropyLoss()
    val_accs = []

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()
            if sched_name == 'OneCycle':
                sched.step()
        if sched_name != 'OneCycle':
            sched.step()

        model.eval()
        with torch.no_grad():
            correct = sum((model(xb).argmax(1) == yb).sum().item() for xb, yb in val_dl)
            val_accs.append(correct / len(val_ds))

    return val_accs

results = {}
for name in ['Constant', 'Cosine', 'OneCycle']:
    results[name] = train_with_schedule(name)
    print(f"{name:>10s}: final val acc = {results[name][-1]:.2%}")

fig, ax = plt.subplots(figsize=(10, 4))
for name, accs in results.items():
    ax.plot(accs, label=name)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Accuracy')
ax.set_title('Effect of LR Schedule on Validation Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Assertion
assert results['Cosine'][-1] > 0.85, "Cosine schedule should achieve decent accuracy"
print("LR scheduling demonstrated ✓")

  Constant: final val acc = 96.85%


    Cosine: final val acc = 96.85%


  OneCycle: final val acc = 96.67%
LR scheduling demonstrated ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_71711/807907026.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
**Understanding OneCycleLR and Super-Convergence**

**Plain language:** OneCycle is like a controlled sprint followed by a cool-down. Instead of always slowing down, you briefly push the learning rate *higher* than normal — this shakes the model out of mediocre solutions — then you decay aggressively to settle into the best one you found. Surprisingly, this "push hard then pull back" approach often reaches the same accuracy in far fewer training steps.

**Building intuition:** The OneCycle policy (Smith & Topin, 2018) has three phases: (1) warmup from a small LR to a peak, (2) a brief period at high LR, and (3) aggressive cosine decay to near-zero. The high-LR phase acts as a form of *regularisation* — the large, noisy updates prevent the model from settling into sharp minima that overfit. Momentum is modulated inversely (low momentum during high LR, high momentum during low LR) to maintain stable updates. The phenomenon where this schedule trains dramatically faster is called *super-convergence*.

**Formal statement:** OneCycleLR defines $\eta_t$ piecewise: warmup phase $\eta_t = \eta_{\min} + (\eta_{\max} - \eta_{\min})\frac{t}{T_1}$ for $t \leq T_1$; decay phase $\eta_t = \eta_{\min} + \frac{\eta_{\max} - \eta_{\min}}{2}(1 + \cos(\pi \frac{t - T_1}{T - T_1}))$ for $t > T_1$. Momentum follows the inverse schedule: $\beta_t = \beta_{\max} - (\beta_{\max} - \beta_{\min})\frac{\eta_t - \eta_{\min}}{\eta_{\max} - \eta_{\min}}$. Super-convergence occurs when this schedule allows training to converge in $\sim$10x fewer iterations than a constant LR baseline.

---

## Structured Interpretation

1. **StepLR** drops the LR by a fixed factor every N epochs. Simple but creates discontinuities that can cause temporary performance drops.

2. **Cosine annealing** smoothly decreases LR following a cosine curve from max to near-zero. No hyperparameters beyond T_max. Popular for its simplicity and strong empirical performance.

3. **Warmup** starts with a very small LR and linearly increases to the target over the first few epochs. This prevents large, destructive gradient updates when the model's weights are random and gradients are noisy.

4. **OneCycleLR** (Smith 2018) combines warmup, a peak, and cosine decay in one cycle. It also modulates momentum inversely. Often achieves the best final performance in fewer epochs — "super-convergence".

5. **For MNEMOSYNE**: We'll use cosine annealing with linear warmup as our default schedule. For short experiments, OneCycleLR is an efficient alternative. The schedule will be specified in the experiment config alongside the optimiser.